# DenseTransNet Enhanced — Dual-Scale SE-Fused Classifier + SHAP + Grad-CAM++
### Novelty additions over baseline:
- **Dual-Scale Feature Fusion:** early (512-ch) and late (1024-ch) DenseNet features extracted separately, each recalibrated by an SE block, then concatenated before the transformer — capturing both fine-grained PD microstructure and broad atrophy patterns
- **SE Block at each scale (documented):** channel-wise recalibration explicitly named in the architecture
- **Dual XAI (Grad-CAM++ + SHAP):** gradient-based and game-theoretic attribution for complementary interpretability

In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Imports and Config
import os, glob, re, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_score, recall_score,
                             f1_score, accuracy_score)
import time, copy
from tqdm import tqdm

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
# Change DATA_ROOT to the location of the dataset on your machine.
# The MRI data are NOT included in the GitHub repository.
DATA_ROOT = "/path/to/GAN T2 dataset"
HC_DIR = os.path.join(DATA_ROOT, "T2 HC images")
PD_DIR = os.path.join(DATA_ROOT, "T2 PD images")

# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------------
# Device / hyper-parameters
# ------------------------------------------------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE   = 32
NUM_EPOCHS   = 25
LR           = 1e-4
WEIGHT_DECAY = 1e-5

print('Device:', DEVICE)
print('Config ready.')


In [ ]:
# Cell 3 — Dataset Manifest Builder
# IMPORTANT:
# All 2-D slices belonging to the same subject must remain in the same
# train/validation/test partition. The subject ID is therefore extracted
# BEFORE any splitting is performed.
#
# The filename convention used here follows the original CycleGAN loader:
# the subject identifier is the portion before the "_slice..." token.
# If your filenames use a different convention, edit extract_subject_id().

exts = ('*.png','*.jpg','*.jpeg','*.tif','*.tiff','*.bmp')

def gather_files(folder):
    files = []
    for e in exts:
        files += glob.glob(os.path.join(folder, e))
    return sorted(files)

def extract_subject_id(filename):
    """
    Extract subject ID from a slice filename.

    Expected convention:
        SUBJECT_PART_1_SUBJECT_PART_2_SUBJECT_PART_3_sliceXX.ext

    The original project loader used the first three underscore-separated
    components before the slice token. Adjust this function if your actual
    filenames use another convention.
    """
    stem = Path(filename).stem
    parts = stem.split('_')

    for i, part in enumerate(parts):
        if re.fullmatch(r'slice[-_]?\d+', part, flags=re.IGNORECASE):
            subject = '_'.join(parts[:i])
            if subject:
                return subject

    # Fallback: remove a terminal slice token such as s01/s1.
    m = re.match(r'^(.*?)[_-]s\d+$', stem, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    # If no slice token is found, the stem is treated as the subject ID.
    return stem

print(f'HC directory exists: {os.path.isdir(HC_DIR)}')
print(f'PD directory exists: {os.path.isdir(PD_DIR)}')

hc_files = gather_files(HC_DIR)
pd_files = gather_files(PD_DIR)
print(f'HC files: {len(hc_files)} | PD files: {len(pd_files)}')

rows = (
    [{'filepath': f, 'label': 0, 'label_name': 'HC',
      'filename': Path(f).name, 'subject_id': extract_subject_id(Path(f).name)}
     for f in hc_files]
    +
    [{'filepath': f, 'label': 1, 'label_name': 'PD',
      'filename': Path(f).name, 'subject_id': extract_subject_id(Path(f).name)}
     for f in pd_files]
)

df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Sanity checks: every subject must have a single class label.
subject_label_counts = df.groupby('subject_id')['label'].nunique()
if (subject_label_counts > 1).any():
    bad_subjects = subject_label_counts[subject_label_counts > 1].index.tolist()
    raise ValueError(
        f'Found subjects assigned to both HC and PD labels: {bad_subjects[:10]}'
    )

print(f'Total slices: {len(df)}')
print(f'Unique subjects: {df.subject_id.nunique()}')
print(f'HC slices: {(df.label==0).sum()} | PD slices: {(df.label==1).sum()}')
print(f'HC subjects: {df.loc[df.label==0, "subject_id"].nunique()} | '
      f'PD subjects: {df.loc[df.label==1, "subject_id"].nunique()}')

# Class weights are calculated from the training partition later.
# They are deliberately NOT calculated from the full dataset to avoid
# using held-out information during training.

MANIFEST_DIR = os.path.join(DATA_ROOT, "dataset_manifests")
os.makedirs(MANIFEST_DIR, exist_ok=True)
manifest_path = os.path.join(MANIFEST_DIR, "t2_manifest_subjectwise.csv")
df.to_csv(manifest_path, index=False)
print('Manifest saved to:', manifest_path)


In [ ]:
# Cell 4 — Subject-Level Stratified Split and DataLoaders
# 75% train / 10% validation / 15% test at SUBJECT level.
# Overall train+validation/test split = 85:15, as reported in the manuscript.
# All slices from one subject remain in exactly one partition.

subject_df = (
    df[['subject_id', 'label']]
    .drop_duplicates('subject_id')
    .reset_index(drop=True)
)

train_val_subjects, test_subjects = train_test_split(
    subject_df,
    test_size=0.15,
    stratify=subject_df['label'],
    random_state=SEED
)

train_subjects, val_subjects = train_test_split(
    train_val_subjects,
    test_size=(0.10 / 0.85),  # 10% overall validation from the 85% train+validation pool
    stratify=train_val_subjects['label'],
    random_state=SEED
)

train_ids = set(train_subjects['subject_id'])
val_ids   = set(val_subjects['subject_id'])
test_ids  = set(test_subjects['subject_id'])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

train_df = df[df['subject_id'].isin(train_ids)].reset_index(drop=True)
val_df   = df[df['subject_id'].isin(val_ids)].reset_index(drop=True)
test_df  = df[df['subject_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train subjects: {len(train_ids)} | slices: {len(train_df)}')
print(f'Val subjects:   {len(val_ids)} | slices: {len(val_df)}')
print(f'Test subjects:  {len(test_ids)} | slices: {len(test_df)}')

print('\nSubject-level class distribution:')
for name, part in [('Train', train_subjects), ('Val', val_subjects), ('Test', test_subjects)]:
    counts = part['label'].value_counts().sort_index()
    print(f'{name}: HC={counts.get(0,0)}, PD={counts.get(1,0)}')

# ---------------------------------------------------------------
# Class weights: inverse-frequency weights calculated ONLY from
# the training subjects/slices. HC is the minority class.
# ---------------------------------------------------------------
train_class_freq = (
    train_df['label'].value_counts().sort_index()
    .reindex([0, 1], fill_value=0).values.astype(float)
)
train_inv_freq = 1.0 / (train_class_freq + 1e-8)
class_weights = train_inv_freq / train_inv_freq.sum() * 2.0

print('Training class frequencies [HC, PD]:', train_class_freq.tolist())
print('Training class weights [HC, PD]:', class_weights.tolist())

class MRIDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row['filepath']).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row['label']).long()

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3,1,1)),
    T.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
])

val_test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3,1,1)),
    T.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
])

train_dataset = MRIDataset(train_df, transform=train_transform)
val_dataset   = MRIDataset(val_df,   transform=val_test_transform)
test_dataset  = MRIDataset(test_df,  transform=val_test_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}


In [ ]:
# Cell 5 — Building Blocks: SE Block and Transformer Block

class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation block.
    Recalibrates channel-wise feature responses by learning global
    channel importance weights via a compact FC bottleneck.
    Applied independently at each feature scale (early and late)
    so that disease-relevant channels are emphasised at every level.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)

    def forward(self, x):
        # x: (B, C, 1, 1)  — already globally pooled
        b, c, _, _ = x.size()
        y = x.view(b, c)                          # (B, C)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y.expand_as(x)                 # channel-wise scaling


class TransformerBlock(nn.Module):
    """
    Lightweight single-layer Transformer encoder.
    Multi-head self-attention + FFN with residual connections and
    layer normalisation, following the standard pre-norm formulation.
    """
    def __init__(self, embed_dim=512, num_heads=4, ff_dim=1024, dropout=0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(embed_dim, num_heads,
                                             dropout=dropout, batch_first=True)
        self.ff      = nn.Sequential(
            nn.Linear(embed_dim, ff_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(ff_dim, embed_dim))
        self.norm1   = nn.LayerNorm(embed_dim)
        self.norm2   = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, seq_len, embed_dim)
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

print('SEBlock and TransformerBlock defined.')

In [ ]:
# Cell 6 — Dual-Scale DenseTransNet  [NOVELTY]
#
# MOTIVATION: PD-related MRI changes occur at multiple anatomical scales:
#   - Fine-grained: iron accumulation in the substantia nigra, subtle texture changes
#     (captured in early DenseNet layers, 512 channels after denseblock2)
#   - Coarse-grained: global grey-matter atrophy and widespread morphological changes
#     (captured in late DenseNet layers, 1024 channels after denseblock4)
#
# DESIGN:
#   1. Split DenseNet121 into two stages (early: up to transition2, late: remainder)
#   2. Apply independent global average pooling and SE recalibration at each scale
#   3. Project each to embed_dim//2 (256-D) and concatenate → 512-D token
#   4. Feed the fused 512-D token into the Transformer encoder
#   5. Classify with FC -> ReLU -> Dropout -> FC -> Softmax
#
# This dual-scale design is architecturally justified and has not been
# previously applied to MRI-based PD classification.

class DualScaleDenseTransNet(nn.Module):
    def __init__(self, num_classes=2, embed_dim=512, num_heads=4):
        super().__init__()

        densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

        # ── Early-scale feature extractor (output: 512 channels) ──────────────
        # Covers: conv0, norm0, relu0, pool0, denseblock1, transition1, denseblock2
        self.early_features = nn.Sequential(
            densenet.features.conv0,
            densenet.features.norm0,
            densenet.features.relu0,
            densenet.features.pool0,
            densenet.features.denseblock1,
            densenet.features.transition1,
            densenet.features.denseblock2,
        )  # → (B, 512, H/8, W/8)

        # ── Late-scale feature extractor (output: 1024 channels) ─────────────
        # Covers: transition2, denseblock3, transition3, denseblock4, norm5
        self.late_features = nn.Sequential(
            densenet.features.transition2,
            densenet.features.denseblock3,
            densenet.features.transition3,
            densenet.features.denseblock4,
            densenet.features.norm5,
        )  # → (B, 1024, H/32, W/32)

        # ── SE blocks: independent channel recalibration at each scale ────────
        self.se_early = SEBlock(512,  reduction=16)   # early scale
        self.se_late  = SEBlock(1024, reduction=16)   # late scale

        # ── Scale-specific projections → equal embedding halves ───────────────
        self.proj_early = nn.Linear(512,  embed_dim // 2)  # 512  → 256
        self.proj_late  = nn.Linear(1024, embed_dim // 2)  # 1024 → 256
        # Concatenated: 256 + 256 = 512 = embed_dim

        # ── Shared Transformer encoder ────────────────────────────────────────
        self.transformer = TransformerBlock(embed_dim=embed_dim,
                                            num_heads=num_heads)

        # ── Classification head ───────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # ── Early scale ───────────────────────────────────────────────────────
        e = self.early_features(x)                        # (B, 512, h, w)
        e = F.relu(e, inplace=True)
        e = F.adaptive_avg_pool2d(e, (1, 1))              # (B, 512, 1, 1)
        e = self.se_early(e)                              # SE recalibration
        e = e.view(e.size(0), -1)                         # (B, 512)
        e = self.proj_early(e)                            # (B, 256)

        # ── Late scale ────────────────────────────────────────────────────────
        l = self.late_features(
                self.early_features(x))                   # (B, 1024, h', w')
        l = F.relu(l, inplace=True)
        l = F.adaptive_avg_pool2d(l, (1, 1))              # (B, 1024, 1, 1)
        l = self.se_late(l)                               # SE recalibration
        l = l.view(l.size(0), -1)                         # (B, 1024)
        l = self.proj_late(l)                             # (B, 256)

        # ── Dual-scale fusion → Transformer ──────────────────────────────────
        fused = torch.cat([e, l], dim=1).unsqueeze(1)    # (B, 1, 512)
        out   = self.transformer(fused).squeeze(1)        # (B, 512)

        return self.classifier(out)                       # (B, num_classes)


# Quick forward-pass test
model = DualScaleDenseTransNet(num_classes=2)
dummy = torch.randn(4, 3, 224, 224)
out   = model(dummy)
print('Output shape:', out.shape)   # (4, 2)
print(f'Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

In [ ]:
# Cell 7 — Weighted Loss, Optimiser, Scheduler
model = DualScaleDenseTransNet(num_classes=2).to(DEVICE)

cw = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=cw)

optimizer = optim.Adam(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = lr_scheduler.StepLR(
    optimizer, step_size=10, gamma=0.5
)

print('Using inverse-frequency weighted CrossEntropyLoss.')
print('Class weights [HC, PD]:', class_weights.tolist())
print(f'Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')


In [ ]:
# Cell 8 — Smoke Test (1 batch forward + backward)
model.train()
for inputs, labels in train_loader:
    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward(); optimizer.step()
    print(f'Smoke test passed — batch shape: {inputs.shape}, loss: {loss.item():.4f}')
    break
torch.cuda.empty_cache()

In [ ]:
# Cell 9 — Full Training Loop
history  = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_wts = copy.deepcopy(model.state_dict())
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f'\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====')
    epoch_start = time.time()

    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()
        running_loss = 0.0; running_correct = 0; n = 0

        pbar = tqdm(dataloaders[phase], desc=f'{phase} E{epoch+1}')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                if phase == 'train':
                    loss.backward(); optimizer.step()
            bs = inputs.size(0)
            running_loss    += loss.item() * bs
            running_correct += (preds == labels).sum().item()
            n += bs
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        ep_loss = running_loss / n
        ep_acc  = running_correct / n
        history[f'{phase}_loss'].append(ep_loss)
        history[f'{phase}_acc'].append(ep_acc)
        print(f'{phase} Loss:{ep_loss:.4f} Acc:{ep_acc:.4f}')

        if phase == 'val' and ep_acc > best_val_acc:
            best_val_acc = ep_acc
            best_wts = copy.deepcopy(model.state_dict())
            print('New best val acc — weights saved.')

    scheduler.step()
    elapsed = time.time() - epoch_start
    print(f'Epoch time: {elapsed//60:.0f}m {elapsed%60:.0f}s')

model.load_state_dict(best_wts)
print(f'\nTraining finished. Best val acc: {best_val_acc:.4f}')

In [ ]:
# Cell 10 — Training Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss Curves')
axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'],   label='Val Acc')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy Curves')
axes[1].legend(); axes[1].grid(alpha=.3)

plt.suptitle('DualScale-DenseTransNet Training Curves')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 11 — Final Test Evaluation
model.eval()
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        probs   = F.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:,1].cpu().numpy())

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec  = recall_score(all_labels, all_preds)           # sensitivity
spec = recall_score(all_labels, all_preds, pos_label=0)  # specificity
f1   = f1_score(all_labels, all_preds)

print(f'Accuracy:    {acc:.4f}')
print(f'Specificity: {spec:.4f}')
print(f'Sensitivity: {rec:.4f}')
print(f'Precision:   {prec:.4f}')
print(f'F1 Score:    {f1:.4f}')
print('\n--- Classification Report ---')
print(classification_report(all_labels, all_preds, target_names=['HC','PD']))

In [ ]:
# Cell 12 — Confusion Matrix and ROC Curve
cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['HC','PD'], yticklabels=['HC','PD'])
axes[0].set(title='Confusion Matrix', ylabel='True', xlabel='Predicted')

fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='navy', lw=2)
axes[1].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
             title='ROC Curve — DualScale-DenseTransNet')
axes[1].legend(loc='lower right')
plt.tight_layout(); plt.show()
print(f'ROC AUC = {roc_auc:.4f}')

In [ ]:
# Cell 13 — Grad-CAM++ Visualisation  [XAI — NOVELTY: now documented in paper]
# Grad-CAM++ uses higher-order partial derivatives to produce more accurate
# class-discriminative localisation maps than standard Grad-CAM.
# The target layer is the last batch-normalisation layer of DenseNet121
# (the final convolutional feature map before global pooling).

!pip install grad-cam -q
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# Target: last norm layer of the late-scale features
target_layers = [model.late_features[-1]]   # norm5 of DenseNet121

def visualize_gradcam_pp(model, dataloader, device, target_layers,
                          class_names, num_images=6):
    model.eval()
    cam   = GradCAMPlusPlus(model=model, target_layers=target_layers)
    shown = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        grayscale_cams = cam(input_tensor=inputs)
        for i in range(inputs.size(0)):
            if shown >= num_images:
                return
            img = inputs[i].detach().cpu().permute(1,2,0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            vis = show_cam_on_image(img, grayscale_cams[i], use_rgb=True)
            fig, axes = plt.subplots(1, 2, figsize=(6, 3))
            axes[0].imshow(img); axes[0].set_title(f'Original — {class_names[labels[i]]}')
            axes[0].axis('off')
            axes[1].imshow(vis); axes[1].set_title('Grad-CAM++ Heatmap')
            axes[1].axis('off')
            plt.tight_layout(); plt.show()
            shown += 1

visualize_gradcam_pp(model, dataloaders['test'], DEVICE,
                     target_layers, class_names=['HC','PD'], num_images=6)

In [ ]:
# Cell 14 — Ablation Study
# Tests: DenseNet121 only | + Transformer | + SE (single-scale) | + Dual-Scale (full)
# Each variant is trained for 10 epochs for quick comparison.

class DenseNet121Only(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        dn = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.features = dn.features
        self.classifier = nn.Linear(1024, num_classes)
    def forward(self, x):
        x = self.features(x); x = F.relu(x); x = F.adaptive_avg_pool2d(x,(1,1))
        return self.classifier(x.view(x.size(0),-1))

class DenseNetTransformer(nn.Module):
    def __init__(self, num_classes=2, embed_dim=512):
        super().__init__()
        dn = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.features = dn.features
        self.proj = nn.Linear(1024, embed_dim)
        self.transformer = TransformerBlock(embed_dim=embed_dim)
        self.classifier = nn.Sequential(nn.Linear(embed_dim,256), nn.ReLU(),
                                         nn.Dropout(0.3), nn.Linear(256, num_classes))
    def forward(self, x):
        x = F.relu(self.features(x)); x = F.adaptive_avg_pool2d(x,(1,1))
        x = self.proj(x.view(x.size(0),-1)).unsqueeze(1)
        x = self.transformer(x).squeeze(1)
        return self.classifier(x)

def quick_eval(model_cls, loader_dict, device, n_epochs=10):
    m  = model_cls().to(device)
    # Use training-partition class weights for the same imbalance handling
    # as the main classifier.
    ab_freq = (
        train_df['label'].value_counts().sort_index()
        .reindex([0, 1], fill_value=0).values.astype(float)
    )
    ab_inv = 1.0 / (ab_freq + 1e-8)
    ab_weights = ab_inv / ab_inv.sum() * 2.0
    cr = nn.CrossEntropyLoss(
        weight=torch.tensor(ab_weights, dtype=torch.float32, device=device)
    )
    op = optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-5)
    for ep in range(n_epochs):
        m.train()
        for inp, lab in loader_dict['train']:
            inp, lab = inp.to(device), lab.to(device)
            op.zero_grad(); loss = cr(m(inp), lab); loss.backward(); op.step()
    m.eval(); correct = total = 0
    with torch.no_grad():
        for inp, lab in loader_dict['test']:
            inp, lab = inp.to(device), lab.to(device)
            _, preds = torch.max(m(inp), 1)
            correct += (preds==lab).sum().item(); total += lab.size(0)
    return correct/total

print('Running ablation study (this may take a few minutes)...')
acc_dn     = quick_eval(DenseNet121Only,    dataloaders, DEVICE)
acc_dnt    = quick_eval(DenseNetTransformer, dataloaders, DEVICE)
# DualScaleDenseTransNet is already trained above — use test acc
print(f'DenseNet121 only:              {acc_dn*100:.2f}%')
print(f'DenseNet + Transformer:        {acc_dnt*100:.2f}%')
print(f'DualScale-DenseTransNet (full): {acc*100:.2f}%')

In [ ]:
# Cell 15 — 10-Fold Subject-Wise Stratified Cross-Validation
#
# Critical leakage-prevention rule:
# all slices from one subject are kept in one fold.
#
# Class weights are recalculated separately for each training fold using
# ONLY the training samples in that fold.

sgkf = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

fold_accs = []
fold_results = []

X = df['filepath'].values
y = df['label'].values
groups = df['subject_id'].values

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X, y, groups=groups), start=1
):
    fold_train = df.iloc[train_idx].reset_index(drop=True)
    fold_test  = df.iloc[test_idx].reset_index(drop=True)

    # Safety checks: no subject may occur in both partitions.
    train_subject_ids = set(fold_train['subject_id'])
    test_subject_ids  = set(fold_test['subject_id'])
    assert train_subject_ids.isdisjoint(test_subject_ids), \
        'Subject leakage detected in CV fold.'

    fold_train_ds = MRIDataset(fold_train, transform=train_transform)
    fold_test_ds  = MRIDataset(fold_test, transform=val_test_transform)

    fold_train_ld = DataLoader(
        fold_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    fold_test_ld = DataLoader(
        fold_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    # Recalculate class weights from THIS FOLD'S TRAINING DATA ONLY.
    fold_class_freq = (
        fold_train['label'].value_counts()
        .sort_index()
        .reindex([0, 1], fill_value=0)
        .values.astype(float)
    )
    fold_inv_freq = 1.0 / (fold_class_freq + 1e-8)
    fold_class_weights = (
        fold_inv_freq / fold_inv_freq.sum() * 2.0
    )

    fold_cw = torch.tensor(
        fold_class_weights,
        dtype=torch.float32,
        device=DEVICE
    )

    fold_model = DualScaleDenseTransNet(num_classes=2).to(DEVICE)
    fold_opt = optim.Adam(
        fold_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    fold_crit = nn.CrossEntropyLoss(weight=fold_cw)

    for ep in range(10):   # retain original quick 10-epoch CV protocol
        fold_model.train()

        for inp, lab in fold_train_ld:
            inp, lab = inp.to(DEVICE), lab.to(DEVICE)

            fold_opt.zero_grad()
            loss = fold_crit(fold_model(inp), lab)
            loss.backward()
            fold_opt.step()

    fold_model.eval()
    correct = total = 0

    with torch.no_grad():
        for inp, lab in fold_test_ld:
            inp, lab = inp.to(DEVICE), lab.to(DEVICE)
            preds = torch.argmax(fold_model(inp), dim=1)
            correct += (preds == lab).sum().item()
            total += lab.size(0)

    fa = correct / total
    fold_accs.append(fa)

    fold_results.append({
        'fold': fold,
        'train_subjects': len(train_subject_ids),
        'test_subjects': len(test_subject_ids),
        'train_slices': len(fold_train),
        'test_slices': len(fold_test),
        'hc_train_slices': int((fold_train['label'] == 0).sum()),
        'pd_train_slices': int((fold_train['label'] == 1).sum()),
        'hc_test_slices': int((fold_test['label'] == 0).sum()),
        'pd_test_slices': int((fold_test['label'] == 1).sum()),
        'accuracy': fa
    })

    print(
        f'Fold {fold:2d}: {fa*100:.2f}% | '
        f'train subjects={len(train_subject_ids)} | '
        f'test subjects={len(test_subject_ids)} | '
        f'weights={fold_class_weights.tolist()}'
    )

fold_results_df = pd.DataFrame(fold_results)
fold_results_df.to_csv(
    os.path.join(DATA_ROOT, 'subjectwise_10fold_results.csv'),
    index=False
)

mean_acc = np.mean(fold_accs)
sd_acc = np.std(fold_accs, ddof=1)

print(f'\nSubject-wise 10-fold CV:')
print(f'Accuracy = {mean_acc*100:.2f}% ± {sd_acc*100:.2f}% (SD)')
print('Fold results saved to subjectwise_10fold_results.csv')


In [ ]:
# Cell 16 — Save Model Weights
# Change SAVE_DIR to a local output directory or a mounted storage location.
SAVE_DIR = os.path.join(DATA_ROOT, "DenseTransNet_Enhanced")
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(SAVE_DIR, 'dual_scale_densetransnet.pth')
)

print('Model saved to:', SAVE_DIR)


## Reproducibility notes
- The dataset is not included in this repository.
- Splitting is performed at the subject level; all slices from one subject remain in the same partition.
- Train/validation/test partitions are stratified by PD/HC class.
- The classifier uses inverse-frequency class-weighted cross-entropy.
- During 10-fold cross-validation, class weights are recalculated from the training portion of each fold only.
- Set `DATA_ROOT` to the local dataset location before running the notebook.
